In [1]:
import zipfile, re
from collections import OrderedDict
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

zip_path = r"C:\Users\Aissatou Ndiaye\Downloads\players_stats_by_season_full_details.csv (1).zip"
out_csv = r"C:\Users\Aissatou Ndiaye\Downloads\vince_carter_per_season.csv"
out_png = r"C:\Users\Aissatou Ndiaye\Downloads\vince_carter_3p_fit.png"

with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()
    csv_name = [n for n in names if n.lower().endswith('.csv')][0]
    df = pd.read_csv(z.open(csv_name), low_memory=False)

df.rename(columns={c: c.strip() for c in df.columns}, inplace=True)
cols_l = [c.lower() for c in df.columns]

def find_col(subs):
    for i,c in enumerate(cols_l):
        for s in subs:
            if s in c:
                return df.columns[i]
    return None

league_col = find_col(['league','lg'])
season_type_col = find_col(['season_type','type','seasontype','gtype','game_type'])
season_col = find_col(['season','year','season_id'])
player_col = find_col(['player','player_name','name'])
three_pct_col = find_col(['3p%','3p_pct','3pt%','threept%','three_p_pct'])
three_made_col = find_col(['3pm','3p_made','threepm','three_p_made','fg3','fg3m'])
three_att_col = find_col(['3pa','3p_att','threepa','three_p_att','fg3a'])
fgm_col = find_col(['fgm','field_goals_made','fieldgoalsmade'])
fga_col = find_col(['fga','field_goals_attempted','fieldgoalsattempted'])

if league_col:
    df_reg = df[df[league_col].astype(str).str.lower().str.contains('nba', na=False)].copy()
else:
    df_reg = df.copy()

if season_type_col:
    df_reg = df_reg[df_reg[season_type_col].astype(str).str.lower().str.contains('regular', na=False)]

def season_to_year(s):
    if pd.isna(s):
        return None
    s = str(s)
    m = re.search(r"(\d{4})", s)
    if m:
        return int(m.group(1))
    m = re.search(r"(\d{2})-(\d{2})", s)
    if m:
        y = int(m.group(1))
        return 2000 + y if y < 50 else 1900 + y
    return None

if player_col is None:
    raise RuntimeError('Player column not found in CSV')

if season_col is None:
    df_reg['season_idx'] = np.arange(len(df_reg))
    season_col = 'season_idx'

df_reg['_season_year'] = df_reg[season_col].apply(season_to_year)

player_season_counts = df_reg.groupby(player_col)['_season_year'].nunique().sort_values(ascending=False)
player_most = player_season_counts.idxmax()

player_df = df_reg[df_reg[player_col]==player_most].copy().dropna(subset=['_season_year']).sort_values('_season_year')

if three_pct_col:
    player_df['three_pct'] = pd.to_numeric(player_df[three_pct_col], errors='coerce')
    if player_df['three_pct'].dropna().max() > 1.1:
        player_df['three_pct'] = player_df['three_pct'] / 100.0
else:
    if three_made_col and three_att_col:
        player_df['three_made'] = pd.to_numeric(player_df[three_made_col], errors='coerce')
        player_df['three_att'] = pd.to_numeric(player_df[three_att_col], errors='coerce')
        player_df['three_pct'] = player_df['three_made'] / player_df['three_att']
    else:
        player_df['three_pct'] = np.nan

if three_made_col:
    player_df['three_made'] = pd.to_numeric(player_df[three_made_col], errors='coerce')
else:
    player_df['three_made'] = np.nan

years = player_df['_season_year'].astype(int).tolist()
all_years = list(range(min(years), max(years)+1))

full = OrderedDict((y, {'three_pct': np.nan, 'three_made': np.nan}) for y in all_years)
for y, row in player_df.set_index('_season_year').iterrows():
    full[int(y)]['three_pct'] = float(row['three_pct']) if not pd.isna(row['three_pct']) else np.nan
    full[int(y)]['three_made'] = float(row['three_made']) if not pd.isna(row['three_made']) else np.nan

all_years_arr = np.array(list(full.keys()))
known_years = np.array([y for y,v in full.items() if not np.isnan(v['three_pct'])])
known_acc = np.array([v['three_pct'] for y,v in full.items() if not np.isnan(v['three_pct'])])
known_made = np.array([v['three_made'] for y,v in full.items() if not np.isnan(v['three_made'])])

if len(known_years) >= 2:
    interp_acc = np.interp(all_years_arr, known_years, known_acc)
    interp_made = np.interp(all_years_arr, known_years, known_made)
else:
    interp_acc = np.array([np.nan]*len(all_years_arr))
    interp_made = np.array([np.nan]*len(all_years_arr))

for y,v in zip(all_years_arr, interp_acc):
    full[int(y)]['three_pct'] = float(v)
for y,v in zip(all_years_arr, interp_made):
    full[int(y)]['three_made'] = float(v)

X = all_years_arr.reshape(-1,1)
y_acc = np.array([full[y]['three_pct'] for y in all_years_arr])
mask = ~np.isnan(y_acc)
model = LinearRegression()
model.fit(X[mask], y_acc[mask])
intercept = model.intercept_
slope = model.coef_[0]

preds = model.predict(X)

f_min = intercept + slope * all_years_arr[0]
f_max = intercept + slope * all_years_arr[-1]
fit_average = (f_min + f_max) / 2

made_vals = np.array([full[y]['three_made'] for y in all_years_arr])
actual_mean_3pm = float(np.nanmean(made_vals))

out_rows = []
for y,pred in zip(all_years_arr, preds):
    out_rows.append({'year': int(y), 'three_pct': full[int(y)]['three_pct'], 'three_made': full[int(y)]['three_made'], 'three_pct_pred': float(pred)})

pd.DataFrame(out_rows).to_csv(out_csv, index=False)

plt.figure(figsize=(10,6))
plt.scatter(all_years_arr, [full[y]['three_pct'] for y in all_years_arr], label='3P% (interpolated)', color='tab:blue')
plt.plot(all_years_arr, preds, color='tab:red', label='Linear fit')
plt.xlabel('Season Start Year')
plt.ylabel('3P%')
plt.title(f"{player_most} 3P% by Season with Linear Fit")
plt.grid(alpha=0.25)
plt.legend()
plt.savefig(out_png, dpi=200)
plt.close()

print('Saved CSV to:', out_csv)
print('Saved plot to:', out_png)
print('Player:', player_most)
print('Seasons:', all_years_arr[0], '-', all_years_arr[-1])
print('Fit average 3P%:', fit_average)
print('Actual average 3PM/season:', actual_mean_3pm)

Saved CSV to: C:\Users\Aissatou Ndiaye\Downloads\vince_carter_per_season.csv
Saved plot to: C:\Users\Aissatou Ndiaye\Downloads\vince_carter_3p_fit.png
Player: Vince Carter
Seasons: 1999 - 2019
Fit average 3P%: 0.35844021652109337
Actual average 3PM/season: 57.523809523809526
